# Study 2: EDI 100 feasibility and equation screen

This notebook searches for an equation set that keeps EDI 100 attainable and maintainable while preserving a measurable pollution and mitigation tradeoff. It follows the Notebook 15 result: at source emissions, sequential development followed by high prevention reached EDI 100; at emissions multiplier 3.12, none of the tested maintenance policies sustained EDI 100.

The screen compares endpoint-matched emissions and income curves, lower high-EDI emissions ceilings, and one-factor changes to soil recovery, prevention effectiveness, and cleanup capacity. It tests target attainment from EDI 0 separately from maintenance at fixed EDI 100 with ample cash. All variants use temporary `EnvironmentConfig` objects in this notebook. The Study 2 environment source, rewards, and configuration defaults are not changed, and no agents are trained.

## 1. Define what counts as feasible

A candidate passes the EDI 100 feasibility gate if:

- at least one development policy reaches EDI 100 and survives the 200-step episode from both 500 and 1000 starting currency per region; and
- at least two maintenance policies survive 200 steps at fixed EDI 100 from both zero pollution and the configured initial pollution of 10.

The screen reports each policy and initial condition separately. It does not treat a pooled success rate as proof of a general balance. Among stress candidates that pass, the later table identifies the least restrictive endpoint emissions factor and reports its peak pollution, spending, and fallbacks. If none pass, the notebook reports that instead of selecting an infeasible equation.

In [1]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.environment import (ACTION_NAMES, EconomicSustainabilityEnvironment, action_parts)

In [2]:
BASE_CONFIG = EnvironmentConfig()
POLLUTANTS = BASE_CONFIG.pollutants
REGIONS = BASE_CONFIG.regions
TARGET_EDI = 100
STRESS_MULTIPLIER = 3.12
GROWTH_CASH_LEVELS = (500, 1000, 2000)
INITIAL_POLLUTION_LEVELS = (0.0, 10.0)
UNLIMITED_CURRENCY = 1_000_000

ACTION_IDS = {}
for action_id, action_name in enumerate(ACTION_NAMES):
    ACTION_IDS[action_name] = action_id

print(f"Target EDI: {TARGET_EDI}")
print(f"Episode limit: {BASE_CONFIG.episode_steps}")
print(f"Stress emissions multiplier: {STRESS_MULTIPLIER}")
print(f"Growth cash levels: {GROWTH_CASH_LEVELS}")

Target EDI: 100
Episode limit: 200
Stress emissions multiplier: 3.12
Growth cash levels: (500, 1000, 2000)


## 2. Specify the candidate equations

For EDI fraction `x = EDI / 100`, the current EDI-linked emissions term is `k_p * 100 * x`. The endpoint-matched curves change how emissions accumulate during development but agree at EDI 100. The endpoint-scaled candidates reduce the linked term's ceiling under the 3.12 stress multiplier. A source multiplier of 3.12 with endpoint fraction about 0.32 has the same EDI-100 linked-emissions endpoint as the current source equation, while retaining a different curve through development.

The income curves also share their EDI-100 value. They test whether front-loading income improves financing during growth. Recovery and mitigation candidates change one configured parameter at a time. These are hypotheses for a synthetic environment, not calibrated empirical laws.

In [3]:
def make_candidate(name, emission_multiplier=1.0, emission_shape="linear", emission_endpoint_fraction=1.0, income_shape="linear", soil_recovery_rate=None, prevention_effectiveness=None,           cleanup_capacity=None):
    return {
        "name": name,
        "emission_multiplier": emission_multiplier,
        "emission_shape": emission_shape,
        "emission_endpoint_fraction": emission_endpoint_fraction,
        "income_shape": income_shape,
        "soil_recovery_rate": soil_recovery_rate,
        "prevention_effectiveness": prevention_effectiveness,
        "cleanup_capacity": cleanup_capacity,
    }

In [4]:
def curve_value(shape, x):
    x = np.clip(float(x), 0.0, 1.0)

    if shape == "linear":
        return x
    if shape == "concave":
        return np.sqrt(x)
    if shape == "quadratic":
        return x ** 2
    if shape == "exponential":
        return np.expm1(2.0 * x) / np.expm1(2.0)

    raise ValueError(f"Unknown curve shape: {shape}")

In [5]:
CANDIDATES = [
    make_candidate("source_linear"),
    make_candidate("source_concave_emissions", emission_shape="concave"),
    make_candidate("source_quadratic_emissions", emission_shape="quadratic"),
    make_candidate("source_exponential_emissions", emission_shape="exponential"),
    make_candidate("source_concave_income", income_shape="concave"),
    make_candidate("source_quadratic_emissions_concave_income", emission_shape="quadratic", income_shape="concave"),
    make_candidate("stress_linear", emission_multiplier=STRESS_MULTIPLIER),
    make_candidate("stress_quadratic", emission_multiplier=STRESS_MULTIPLIER, emission_shape="quadratic"),
    make_candidate("stress_exponential", emission_multiplier=STRESS_MULTIPLIER, emission_shape="exponential"),
    make_candidate("stress_scaled_endpoint_0.25", emission_multiplier=STRESS_MULTIPLIER, emission_endpoint_fraction=0.25),
    make_candidate("stress_scaled_endpoint_0.30", emission_multiplier=STRESS_MULTIPLIER, emission_endpoint_fraction=0.30),
    make_candidate("stress_scaled_endpoint_0.32", emission_multiplier=STRESS_MULTIPLIER, emission_endpoint_fraction=0.32),
    make_candidate("stress_scaled_endpoint_0.35", emission_multiplier=STRESS_MULTIPLIER, emission_endpoint_fraction=0.35),
    make_candidate("stress_scaled_endpoint_0.40", emission_multiplier=STRESS_MULTIPLIER, emission_endpoint_fraction=0.40),
    make_candidate("stress_scaled_endpoint_0.32_quadratic_income", emission_multiplier=STRESS_MULTIPLIER, emission_endpoint_fraction=0.32, emission_shape="quadratic", income_shape="quadratic"),
    make_candidate("stress_soil_recovery_0.04", emission_multiplier=STRESS_MULTIPLIER, soil_recovery_rate=0.04),
    make_candidate("stress_soil_recovery_0.06", emission_multiplier=STRESS_MULTIPLIER, soil_recovery_rate=0.06),
    make_candidate("stress_prevention_0.85", emission_multiplier=STRESS_MULTIPLIER, prevention_effectiveness=0.85),
    make_candidate("stress_prevention_0.92", emission_multiplier=STRESS_MULTIPLIER, prevention_effectiveness=0.92),
    make_candidate("stress_cleanup_capacity_15", emission_multiplier=STRESS_MULTIPLIER, cleanup_capacity=15.0),
    make_candidate("stress_cleanup_capacity_20", emission_multiplier=STRESS_MULTIPLIER, cleanup_capacity=20.0),
]

In [6]:
rows = []
for candidate in CANDIDATES:
    soil_recovery = candidate["soil_recovery_rate"]
    prevention = candidate["prevention_effectiveness"]
    cleanup = candidate["cleanup_capacity"]

    if soil_recovery is None:
        soil_recovery = BASE_CONFIG.natural_recovery_rates["soil"]

    if prevention is None:
        prevention = BASE_CONFIG.prevention_fractions[3]

    if cleanup is None:
        cleanup = BASE_CONFIG.cleanup_capacities[3]

    row = {
        "candidate": candidate["name"],
        "emissions multiplier": candidate["emission_multiplier"],
        "emissions curve": candidate["emission_shape"],
        "EDI-100 linked-emissions factor": candidate["emission_multiplier"] * candidate["emission_endpoint_fraction"],
        "income curve": candidate["income_shape"],
        "soil recovery rate": soil_recovery,
        "high prevention effectiveness": prevention,
        "high cleanup capacity": cleanup,
    }
    rows.append(row)

candidate_table = pd.DataFrame(rows)
display(candidate_table.round(3))

,candidate,emissions multiplier,emissions curve,EDI-100 linked-emissions factor,income curve,soil recovery rate,high prevention effectiveness,high cleanup capacity
0,source_linear,1.00,linear,1.000,linear,0.02,0.75,10.0
1,source_concave_emissions,1.00,concave,1.000,linear,0.02,0.75,10.0
2,source_quadratic_emissions,1.00,quadratic,1.000,linear,0.02,0.75,10.0
3,source_exponential_emissions,1.00,exponential,1.000,linear,0.02,0.75,10.0
4,source_concave_income,1.00,linear,1.000,concave,0.02,0.75,10.0
5,source_quadratic_emissions_concave_income,1.00,quadratic,1.000,concave,0.02,0.75,10.0
6,stress_linear,3.12,linear,3.120,linear,0.02,0.75,10.0
7,stress_quadratic,3.12,quadratic,3.120,linear,0.02,0.75,10.0
8,stress_exponential,3.12,exponential,3.120,linear,0.02,0.75,10.0
9,stress_scaled_endpoint_0.25,3.12,linear,0.780,linear,0.02,0.75,10.0


### Why soil deserves a separate feasibility check

Below its limit, source soil recovery removes `0.02 * S * (1 - 0.8 * S / 100)` units per region per step, where `S` is current soil stock. This removal curve peaks at 0.625 units when `S = 62.5`. At EDI 100 and emissions multiplier 3.12, gross soil generation is `0.5 + 0.02 * 100 * 3.12 = 6.74`; high prevention leaves 1.685. Prevention alone therefore generates soil faster than the source recovery equation can remove it at every below-limit stock. Cleanup or a different emissions or recovery equation is needed to balance that flow.

This calculation is a flow check for the high-prevention action. It does not prove that every sequence of prevention and cleanup actions is infeasible.

## 3. Check one high-cleanup transition before the search

Notebook 15 reported zero pollution under source emissions, EDI 100, and high cleanup from zero initial pollution. That conflicts with source air emissions of 11 per region per step and high cleanup capacity of 10. This deterministic one-step check reports the environment's raw air emissions, cleanup removal, and resulting stock so the later maintenance results can be interpreted against the actual transition.

In [7]:
audit_config = EnvironmentConfig()
audit_environment = EconomicSustainabilityEnvironment(config=audit_config)
audit_environment.reset(61_000)
audit_environment.development_indices = (100, 100, 100, 100)
audit_environment.currency_balances = (1_000_000, 1_000_000, 1_000_000, 1_000_000)
audit_environment.pollutant_loads = {pollutant: (0.0, 0.0, 0.0, 0.0) for pollutant in POLLUTANTS}
audit_environment._update_indices()

audit_actions = [ACTION_IDS["maintain_clean_high"]] * len(REGIONS)
_, _, audit_terminated, _, audit_info = audit_environment.step(audit_actions)

audit_rows = []
for region in REGIONS:
    audit_rows.append({
        "region": region,
        "action": audit_info["action_names"][region],
        "gross air emissions": audit_info["gross_emissions"]["air"][region],
        "air cleanup removed": audit_info["cleanup_removed"]["air"][region],
        "air stock after cleanup": audit_info["pollutant_loads"]["air"][region],
        "failed": audit_terminated,
    })

display(pd.DataFrame(audit_rows).round(3))
print(f"Configured high cleanup capacity: {audit_config.cleanup_capacities[3]}")

,region,action,gross air emissions,air cleanup removed,air stock after cleanup,failed
0,0,maintain_clean_high,11.0,10.0,1.0,False
1,1,maintain_clean_high,11.0,10.0,1.0,False
2,2,maintain_clean_high,11.0,10.0,1.0,False
3,3,maintain_clean_high,11.0,10.0,1.0,False


Configured high cleanup capacity: 10.0


## 4. Run candidate equations through the actual environment

The environment uses one shared per-EDI coefficient for all regions. To test nonlinear curves without changing `environment.py`, this notebook starts all four regions with identical EDI, currency, and pollution, and uses policies that keep their states synchronized. Before each step it updates a temporary configuration coefficient so the unchanged linear environment equation produces the candidate curve exactly. The symmetry check is recorded for every episode; any nonzero divergence would invalidate that curve implementation for that run.

In [8]:
def make_candidate_config(candidate):
    config = EnvironmentConfig()

    if candidate["soil_recovery_rate"] is not None:
        recovery_rates = dict(config.natural_recovery_rates)
        recovery_rates["soil"] = candidate["soil_recovery_rate"]
        config.natural_recovery_rates = recovery_rates

    if candidate["prevention_effectiveness"] is not None:
        prevention_fractions = list(config.prevention_fractions)
        prevention_fractions[3] = candidate["prevention_effectiveness"]
        config.prevention_fractions = tuple(prevention_fractions)

    if candidate["cleanup_capacity"] is not None:
        cleanup_capacities = list(config.cleanup_capacities)
        cleanup_capacities[3] = candidate["cleanup_capacity"]
        config.cleanup_capacities = tuple(cleanup_capacities)

    return config

In [9]:
def update_candidate_equations(config, candidate, edi):
    x = np.clip(edi / 100.0, 0.0, 1.0)
    emission_curve = curve_value(candidate["emission_shape"], x)
    emission_curve *= candidate["emission_endpoint_fraction"]
    income_curve = curve_value(candidate["income_shape"], x)

    if x > 0:
        emission_rate_scale = (
            candidate["emission_multiplier"] * emission_curve / x)
        income_rate_scale = income_curve / x
    else:
        emission_rate_scale = (candidate["emission_multiplier"] * candidate["emission_endpoint_fraction"])
        income_rate_scale = 1.0

    config.edi_emissions_per_point = {pollutant: BASE_CONFIG.edi_emissions_per_point[pollutant] * emission_rate_scale for pollutant in POLLUTANTS}
    config.income_per_edi = BASE_CONFIG.income_per_edi * income_rate_scale

In [10]:
def action_cost(action_name, config):
    action_id = ACTION_IDS[action_name]
    development_choice, investment_choice = action_parts(action_id)
    cost = config.development_cost if development_choice else 0.0

    if investment_choice in (1, 2, 3):
        cost += config.prevention_costs[investment_choice]
    elif investment_choice in (4, 5, 6):
        cleanup_level = investment_choice - 3
        cost += config.cleanup_costs[cleanup_level]

    return cost

In [11]:
def action_is_affordable(action_name, balance, edi, config):
    action_id = ACTION_IDS[action_name]
    development_choice, _ = action_parts(action_id)
    can_develop = not development_choice or edi < 100
    return can_develop and balance >= action_cost(action_name, config)

In [12]:
def regional_pressure(environment, region):
    pressures = []
    for pollutant in POLLUTANTS:
        load = environment.pollutant_loads[pollutant][region]
        limit = environment.config.pollution_limits[pollutant]
        pressure = load / limit

        pressures.append(pressure)

    return max(pressures)

In [13]:
def choose_candidate_actions(policy_name, environment, target):
    rule = POLICY_RULES[policy_name]
    proposed_actions = []
    executed_actions = []
    fallback_count = 0

    for region in REGIONS:
        edi = environment.development_indices[region]
        balance = environment.currency_balances[region]
        developing = edi < target
        pressure = regional_pressure(environment, region)

        kind = rule["kind"]

        if kind == "sequential":
            preferred = "maintain_prevent_high"
            if developing:
                preferred = "develop_none"

        elif kind == "medium_then_high":
            preferred = "maintain_prevent_high"
            if developing:
                preferred = "develop_prevent_medium"

        elif kind == "fixed":
            preferred = rule["action"]

        else:
            investment = "prevent_high"
            if pressure >= rule["threshold"]:
                investment = "clean_high"

            activity = "maintain"
            if developing:
                activity = "develop"
            preferred = f"{activity}_{investment}"


        if rule["kind"] == "sequential" and not action_is_affordable(preferred, balance, edi, environment.config):
            executed = "maintain_none"
        else:
            if preferred.startswith("develop_"):
                fallback_order = [
                    preferred,
                    "develop_none",
                    "develop_prevent_medium",
                    "develop_prevent_high",
                    "develop_clean_high",
                    "maintain_prevent_high",
                    "maintain_none",
                ]
            else:
                fallback_order = [
                    preferred,
                    "maintain_prevent_high",
                    "maintain_clean_high",
                    "maintain_prevent_medium",
                    "maintain_clean_medium",
                    "maintain_none",
                ]

            executed = "maintain_none"
            for action_name in dict.fromkeys(fallback_order):
                if action_is_affordable(action_name, balance, edi, environment.config):
                    executed = action_name
                    break

        proposed_actions.append(ACTION_IDS[preferred])
        executed_actions.append(ACTION_IDS[executed])
        fallback_count += executed != preferred

    return proposed_actions, executed_actions, fallback_count

In [14]:
def run_candidate_episode(candidate, policy_name, initial_edi, initial_cash, initial_pollution, save_trace=False):
    config = make_candidate_config(candidate)
    environment = EconomicSustainabilityEnvironment(config=config)
    environment.reset(62_000)
    environment.development_indices = (initial_edi,) * len(REGIONS)
    environment.currency_balances = (initial_cash,) * len(REGIONS)
    environment.pollutant_loads = {pollutant: (initial_pollution,) * len(REGIONS) for pollutant in POLLUTANTS}
    environment._update_indices()

    income_total = 0.0
    spending_total = 0.0
    development_spending = 0.0
    prevention_spending = 0.0
    cleanup_spending = 0.0
    fallback_count = 0
    target_reached_step = np.nan
    peak_pollution = {pollutant: max(environment.pollutant_loads[pollutant]) for pollutant in POLLUTANTS}
    peak_pressure = max(regional_pressure(environment, region) for region in REGIONS)
    gross_emissions_total = {pollutant: 0.0 for pollutant in POLLUTANTS}
    cleanup_removed_total = {pollutant: 0.0 for pollutant in POLLUTANTS}
    trajectory = []
    symmetry_error = 0.0

    for step in range(1, config.episode_steps + 1):
        current_edi = environment.development_indices[0]
        update_candidate_equations(config, candidate, current_edi)
        proposed_actions, executed_actions, step_fallbacks = choose_candidate_actions(policy_name, environment, TARGET_EDI)
        _, _, terminated, truncated, info = environment.step(executed_actions)
        fallback_count += step_fallbacks
        income_total += sum(info["income"])
        spending_total += sum(info["costs_paid"])
        development_spending += sum(info["development_costs_paid"])
        prevention_spending += sum(info["prevention_costs_paid"])
        cleanup_spending += sum(info["cleanup_costs_paid"])

        for pollutant in POLLUTANTS:
            gross_emissions_total[pollutant] += sum(info["gross_emissions"][pollutant])
            cleanup_removed_total[pollutant] += sum(info["cleanup_removed"][pollutant])
            peak_pollution[pollutant] = max(peak_pollution[pollutant], max(info["pollutant_loads"][pollutant]))

        for pollutant in POLLUTANTS:
            for region in REGIONS:
                load = info["pollutant_loads"][pollutant][region]
                limit = config.pollution_limits[pollutant]
                pressure = load / limit

                peak_pressure = max(peak_pressure, pressure)
        symmetry_error = max(symmetry_error, max(environment.development_indices) - min(environment.development_indices), max(environment.currency_balances) - min(environment.currency_balances))

        if min(environment.development_indices) >= TARGET_EDI and np.isnan(target_reached_step):
            target_reached_step = step


        if save_trace:
            for region in REGIONS:
                trace_row = {
                    "step": step,
                    "region": region,
                    "proposed_action": ACTION_NAMES[proposed_actions[region]],
                    "executed_action": info["action_names"][region],
                    "edi": environment.development_indices[region],
                    "currency": environment.currency_balances[region],
                    "income": info["income"][region],
                    "spending": info["costs_paid"][region],
                    "pressure": regional_pressure(environment, region),
                }

                for pollutant in POLLUTANTS:
                    gross_emissions = info["gross_emissions"][pollutant][region]
                    cleanup_removed = info["cleanup_removed"][pollutant][region]
                    pollutant_load = info["pollutant_loads"][pollutant][region]

                    trace_row[f"gross_{pollutant}"] = gross_emissions
                    trace_row[f"cleanup_{pollutant}"] = cleanup_removed
                    trace_row[pollutant] = pollutant_load

                trajectory.append(trace_row)

        if terminated or truncated:
            break

    reached_target = min(environment.development_indices) >= TARGET_EDI

    failure_step = np.nan
    failure_pollutants = ""

    if terminated:
        failure_step = environment.step_count
        failed_pollutants = set()

        for region, pollutant in info["failed_regions"]:
            failed_pollutants.add(pollutant)
        failure_pollutants = ", ".join(sorted(failed_pollutants))

    row = {
        "candidate": candidate["name"],
        "policy": policy_name,
        "initial_edi": initial_edi,
        "initial_cash": initial_cash,
        "initial_pollution": initial_pollution,
        "steps": environment.step_count,
        "failed": terminated,
        "survived": not terminated,
        "reached_target": reached_target,
        "success": not terminated and reached_target,
        "target_reached_step": target_reached_step,
        "failure_step": failure_step,
        "failure_pollutants": failure_pollutants,
        "final_edi": float(np.mean(environment.development_indices)),
        "final_currency": float(np.mean(environment.currency_balances)),
        "income": income_total,
        "spending": spending_total,
        "development_spending": development_spending,
        "prevention_spending": prevention_spending,
        "cleanup_spending": cleanup_spending,
        "fallback_count": fallback_count,
        "peak_pressure": peak_pressure,
        "peak_pollution": peak_pollution,
        "gross_emissions": gross_emissions_total,
        "cleanup_removed": cleanup_removed_total,
        "symmetry_error": symmetry_error,
    }

    if save_trace:
        row["trajectory"] = trajectory

    return row

In [15]:
POLICY_RULES = {
    "sequential_high_prevention": {"kind": "sequential"},
    "develop_medium_then_high": {"kind": "medium_then_high"},
    "adaptive_0.25": {"kind": "adaptive", "threshold": 0.25},
    "adaptive_0.50": {"kind": "adaptive", "threshold": 0.50},
    "maintain_prevent_high": {"kind": "fixed", "action": "maintain_prevent_high"},
    "maintain_clean_high": {"kind": "fixed", "action": "maintain_clean_high"},
}
GROWTH_POLICIES = (
    "sequential_high_prevention",
    "develop_medium_then_high",
    "adaptive_0.25",
    "adaptive_0.50",
)
MAINTENANCE_POLICIES = (
    "maintain_prevent_high",
    "maintain_clean_high",
    "adaptive_0.25",
    "adaptive_0.50",
)

## 5. Test maintenance feasibility at fixed EDI 100

This probe uses ample currency, initial pollution of 0 or 10, and four maintenance policies. The goal is to isolate whether each candidate equation can maintain EDI 100 once development and starting-cash limits are removed. It also records gross emissions and cleanup removal so the observed pollution stocks can be checked against the action capacities.

In [16]:
maintenance_rows = []
for candidate, policy_name, initial_pollution in product(CANDIDATES, MAINTENANCE_POLICIES, INITIAL_POLLUTION_LEVELS):
    maintenance_rows.append(run_candidate_episode(
        candidate,
        policy_name,
        initial_edi=TARGET_EDI,
        initial_cash=UNLIMITED_CURRENCY,
        initial_pollution=initial_pollution,
    ))

maintenance_results = pd.DataFrame(maintenance_rows)
maintenance_summary = maintenance_results[[
    "candidate", "policy", "initial_pollution", "survived", "steps",
    "failure_step", "failure_pollutants", "peak_pressure", "peak_pollution",
    "gross_emissions", "cleanup_removed", "prevention_spending",
    "cleanup_spending", "symmetry_error",
]]
display(maintenance_summary.round(3))

,candidate,policy,initial_pollution,survived,steps,failure_step,failure_pollutants,peak_pressure,peak_pollution,gross_emissions,cleanup_removed,prevention_spending,cleanup_spending,symmetry_error
0,source_linear,maintain_prevent_high,0.0,True,200,NaN,,0.417,"{'air': 4.637089838765283, 'water': 11.3124705...","{'air': 8800.0, 'water': 4000.0, 'soil': 2000....","{'air': 0.0, 'water': 0.0, 'soil': 0.0, 'noise...",40000.0,0.0,0.0
1,source_linear,maintain_prevent_high,10.0,True,200,NaN,,0.430,"{'air': 10.0, 'water': 11.312470604254367, 'so...","{'air': 8800.0, 'water': 4000.0, 'soil': 2000....","{'air': 0.0, 'water': 0.0, 'soil': 0.0, 'noise...",40000.0,0.0,0.0
2,source_linear,maintain_clean_high,0.0,True,200,NaN,,0.017,"{'air': 1.673669591419774, 'water': 0.0, 'soil...","{'air': 8800.0, 'water': 4000.0, 'soil': 2000....","{'air': 8000.0, 'water': 4000.0, 'soil': 2000....",0.0,64000.0,0.0
3,source_linear,maintain_clean_high,10.0,True,200,NaN,,0.100,"{'air': 10.0, 'water': 10.0, 'soil': 10.0, 'no...","{'air': 8800.0, 'water': 4000.0, 'soil': 2000....","{'air': 8000.0, 'water': 4033.4088893152066, '...",0.0,64000.0,0.0
4,source_linear,adaptive_0.25,0.0,True,200,NaN,,0.252,"{'air': 4.637089838765283, 'water': 11.3043010...","{'air': 8800.0, 'water': 4000.0, 'soil': 2000....","{'air': 240.0, 'water': 222.34149099404135, 's...",38800.0,1920.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,stress_cleanup_capacity_20,maintain_clean_high,10.0,True,200,NaN,,0.137,"{'air': 13.738535028489231, 'water': 10.0, 'so...","{'air': 22368.000000000025, 'water': 10784.000...","{'air': 16000.0, 'water': 10817.398917965349, ...",0.0,64000.0,0.0
164,stress_cleanup_capacity_20,adaptive_0.25,0.0,True,200,NaN,,0.258,"{'air': 12.987391113412485, 'water': 25.800609...","{'air': 22368.000000000025, 'water': 10784.000...","{'air': 2480.0, 'water': 2024.6180846619184, '...",33800.0,9920.0,0.0
165,stress_cleanup_capacity_20,adaptive_0.25,10.0,True,200,NaN,,0.258,"{'air': 12.987391113412485, 'water': 25.800607...","{'air': 22368.000000000025, 'water': 10784.000...","{'air': 2560.0, 'water': 2090.4972774122184, '...",33600.0,10240.0,0.0
166,stress_cleanup_capacity_20,adaptive_0.50,0.0,True,200,NaN,,0.510,"{'air': 12.987364551006287, 'water': 35.749707...","{'air': 22368.000000000025, 'water': 10784.000...","{'air': 1920.0, 'water': 1540.971210047778, 's...",35200.0,7680.0,0.0


## 6. Test reaching EDI 100 from a matched starting state

Each candidate is tested with four non-learning development policies and the same three starting-cash levels. Every region starts with the same cash and pollution, so the nonlinear emissions and income curves can be represented exactly through a shared temporary config coefficient. The simulator reports a symmetry error; it should remain zero. Starting cash 2000 is a diagnostic stress level above the source configuration maximum, included to show whether liquidity alone can unlock the target.

In [17]:
GROWTH_CASH_LEVELS = (500, 1000, 2000)
growth_rows = []
for candidate, policy_name, initial_cash in product(CANDIDATES, GROWTH_POLICIES, GROWTH_CASH_LEVELS):
    growth_rows.append(run_candidate_episode(candidate, policy_name, initial_edi=0, initial_cash=initial_cash, initial_pollution=10.0))

growth_results = pd.DataFrame(growth_rows)
growth_summary = growth_results[[
    "candidate", "policy", "initial_cash", "success", "survived",
    "reached_target", "target_reached_step", "failure_step",
    "failure_pollutants", "final_edi", "final_currency", "income",
    "development_spending", "prevention_spending", "cleanup_spending",
    "fallback_count", "peak_pressure", "symmetry_error",
]]
display(growth_summary.round(3))

,candidate,policy,initial_cash,success,survived,reached_target,target_reached_step,failure_step,failure_pollutants,final_edi,final_currency,income,development_spending,prevention_spending,cleanup_spending,fallback_count,peak_pressure,symmetry_error
0,source_linear,sequential_high_prevention,500,True,True,True,39.0,NaN,,100.0,3347.5,53590.0,10000.0,32200.0,0.0,56,0.662,0.0
1,source_linear,sequential_high_prevention,1000,True,True,True,29.0,NaN,,100.0,3497.5,54190.0,10000.0,34200.0,0.0,16,0.548,0.0
2,source_linear,sequential_high_prevention,2000,True,True,True,25.0,NaN,,100.0,4317.5,54270.0,10000.0,35000.0,0.0,0,0.511,0.0
3,source_linear,develop_medium_then_high,500,False,True,False,NaN,NaN,,45.0,87.5,33750.0,2000.0,33400.0,0.0,784,0.327,0.0
4,source_linear,develop_medium_then_high,1000,True,True,True,75.0,NaN,,100.0,2587.5,52450.0,10000.0,36100.0,0.0,256,0.498,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,stress_cleanup_capacity_20,adaptive_0.25,1000,False,False,False,NaN,173.0,soil,60.0,85.0,34140.0,3200.0,34600.0,0.0,660,1.000,0.0
248,stress_cleanup_capacity_20,adaptive_0.25,2000,True,True,True,38.0,NaN,,100.0,2269.5,54078.0,10000.0,31800.0,11200.0,100,0.732,0.0
249,stress_cleanup_capacity_20,adaptive_0.50,500,False,False,False,NaN,137.0,soil,40.0,80.0,21720.0,1600.0,21800.0,0.0,536,1.025,0.0
250,stress_cleanup_capacity_20,adaptive_0.50,1000,False,False,False,NaN,173.0,soil,60.0,85.0,34140.0,3200.0,34600.0,0.0,660,1.000,0.0


## 7. Apply the feasibility gate and rank passing stress candidates

For each candidate, count growth policies that succeed at both 500 and 1000 initial currency, then count maintenance policies that survive from both pollution starting levels. A stress candidate passes if at least one growth policy and at least two maintenance policies meet those conditions. The highest endpoint emissions factor among passing stress candidates is the least restrictive feasible endpoint in this screen. Pollution peaks, spending, and fallback counts remain visible for the tradeoff discussion; the gate does not select a candidate just because it makes pollution negligible.

In [18]:
growth_cash_pivot = growth_results.pivot_table(index=["candidate", "policy"], columns="initial_cash", values="success", aggfunc="mean")
growth_policy_pass = ((growth_cash_pivot[500] == 1.0) & (growth_cash_pivot[1000] == 1.0)).groupby("candidate").sum().rename("growth_policies_pass")

maintenance_stock_pivot = maintenance_results.pivot_table(index=["candidate", "policy"], columns="initial_pollution", values="survived", aggfunc="mean")
maintenance_policy_pass = ((maintenance_stock_pivot[0.0] == 1.0) & (maintenance_stock_pivot[10.0] == 1.0)).groupby("candidate").sum().rename("maintenance_policies_pass")

feasibility_summary = pd.concat([growth_policy_pass, maintenance_policy_pass], axis=1).reset_index()
feasibility_summary = feasibility_summary.merge(candidate_table[["candidate", "emissions multiplier", "emissions curve", "EDI-100 linked-emissions factor", "income curve",
                     "soil recovery rate", "high prevention effectiveness", "high cleanup capacity"]], on="candidate")

feasibility_summary["feasible"] = ((feasibility_summary["growth_policies_pass"] >= 1) & (feasibility_summary["maintenance_policies_pass"] >= 2))

stress_feasibility = feasibility_summary[feasibility_summary["emissions multiplier"] == STRESS_MULTIPLIER].sort_values(["feasible", "EDI-100 linked-emissions factor"], ascending=[False, False])
display(stress_feasibility.round(3))

,candidate,growth_policies_pass,maintenance_policies_pass,emissions multiplier,emissions curve,EDI-100 linked-emissions factor,income curve,soil recovery rate,high prevention effectiveness,high cleanup capacity,feasible
17,stress_scaled_endpoint_0.35,1,4,3.12,linear,1.092,linear,0.02,0.75,10.0,True
15,stress_scaled_endpoint_0.32,1,4,3.12,linear,0.998,linear,0.02,0.75,10.0,True
16,stress_scaled_endpoint_0.32_quadratic_income,1,4,3.12,quadratic,0.998,quadratic,0.02,0.75,10.0,True
14,stress_scaled_endpoint_0.30,1,4,3.12,linear,0.936,linear,0.02,0.75,10.0,True
13,stress_scaled_endpoint_0.25,1,4,3.12,linear,0.780,linear,0.02,0.75,10.0,True
6,stress_cleanup_capacity_15,0,3,3.12,linear,3.120,linear,0.02,0.75,15.0,False
7,stress_cleanup_capacity_20,0,3,3.12,linear,3.120,linear,0.02,0.75,20.0,False
8,stress_exponential,0,0,3.12,exponential,3.120,linear,0.02,0.75,10.0,False
9,stress_linear,0,0,3.12,linear,3.120,linear,0.02,0.75,10.0,False
10,stress_prevention_0.85,0,0,3.12,linear,3.120,linear,0.02,0.85,10.0,False


In [19]:
passing_candidates = stress_feasibility[stress_feasibility["feasible"]]
if passing_candidates.empty:
    print("No stress candidate passed the stated EDI-100 gate. Keep the source equation as the feasible reference and widen the candidate search before selecting a stress equation.")
else:
    least_restrictive = passing_candidates.sort_values("EDI-100 linked-emissions factor", ascending=False).iloc[0]
    print("Least restrictive passing stress candidate:")
    display(least_restrictive.to_frame("value"))

    selected_name = least_restrictive["candidate"]
    selected_growth = growth_results[growth_results["candidate"] == selected_name]
    selected_maintenance = maintenance_results[maintenance_results["candidate"] == selected_name]
    print("Growth outcomes for the selected candidate:")
    display(selected_growth[["policy", "initial_cash", "success", "final_edi", "failure_step", "fallback_count", "peak_pressure", "spending"]].round(3))

    print("Maintenance outcomes for the selected candidate:")
    display(selected_maintenance[["policy", "initial_pollution", "survived", "failure_step", "failure_pollutants", "peak_pressure", "peak_pollution","gross_emissions", "cleanup_removed"]].round(3))

Least restrictive passing stress candidate:


,value
candidate,stress_scaled_endpoint_0.35
growth_policies_pass,1
maintenance_policies_pass,4
emissions multiplier,3.12
emissions curve,linear
EDI-100 linked-emissions factor,1.092
income curve,linear
soil recovery rate,0.02
high prevention effectiveness,0.75
high cleanup capacity,10.0


Growth outcomes for the selected candidate:


,policy,initial_cash,success,final_edi,failure_step,fallback_count,peak_pressure,spending
144,sequential_high_prevention,500,True,100.0,NaN,56,0.828,42200.0
145,sequential_high_prevention,1000,True,100.0,NaN,16,0.642,44200.0
146,sequential_high_prevention,2000,True,100.0,NaN,0,0.589,45000.0
147,develop_medium_then_high,500,False,45.0,NaN,784,0.353,35400.0
148,develop_medium_then_high,1000,True,100.0,NaN,256,0.566,46100.0
149,develop_medium_then_high,2000,True,100.0,NaN,0,0.496,47500.0
150,adaptive_0.25,500,False,40.0,NaN,788,0.361,33600.0
151,adaptive_0.25,1000,False,60.0,NaN,768,0.277,43200.0
152,adaptive_0.25,2000,True,100.0,NaN,80,0.312,49880.0
153,adaptive_0.50,500,False,40.0,NaN,788,0.361,33600.0


Maintenance outcomes for the selected candidate:


,policy,initial_pollution,survived,failure_step,failure_pollutants,peak_pressure,peak_pollution,gross_emissions,cleanup_removed
96,maintain_prevent_high,0.0,True,NaN,,0.464,"{'air': 4.951288135501903, 'water': 12.2304146...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 0.0, 'water': 0.0, 'soil': 0.0, 'noise..."
97,maintain_prevent_high,10.0,True,NaN,,0.478,"{'air': 10.0, 'water': 12.230414625225343, 'so...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 0.0, 'water': 0.0, 'soil': 0.0, 'noise..."
98,maintain_clean_high,0.0,True,NaN,,0.029,"{'air': 2.914570130953953, 'water': 0.0, 'soil...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 8000.0, 'water': 4294.400000000016, 's..."
99,maintain_clean_high,10.0,True,NaN,,0.100,"{'air': 10.0, 'water': 10.0, 'soil': 10.0, 'no...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 8000.0, 'water': 4327.518203108917, 's..."
100,adaptive_0.25,0.0,True,NaN,,0.253,"{'air': 4.951288135501903, 'water': 12.2077164...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 280.0, 'water': 258.0252980091597, 'so..."
101,adaptive_0.25,10.0,True,NaN,,0.252,"{'air': 10.0, 'water': 12.203575770612439, 'so...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 320.0, 'water': 294.9128470328908, 'so..."
102,adaptive_0.50,0.0,True,NaN,,0.464,"{'air': 4.951288135501903, 'water': 12.2304146...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 0.0, 'water': 0.0, 'soil': 0.0, 'noise..."
103,adaptive_0.50,10.0,True,NaN,,0.478,"{'air': 10.0, 'water': 12.230414625225343, 'so...","{'air': 9388.800000000021, 'water': 4294.40000...","{'air': 0.0, 'water': 0.0, 'soil': 0.0, 'noise..."


## 8. How to choose the equation

Prefer a candidate that passes the EDI-100 gate across distinct growth and maintenance policies, then compare its pollutant-specific peak loads, episode duration, mitigation spending, and cash fallbacks. A candidate that only works at initial currency 2000 is a liquidity-dependent result, not a robust choice. A candidate that passes only because its emissions endpoint is extremely low should be compared with recovery and mitigation alternatives before selection.

The source linear equation is the reference and should remain in the comparison even if it is not selected for a stress case. Endpoint-matched curve shapes can change the path to EDI 100 but cannot change the emissions level during fixed-EDI maintenance. Endpoint-scaled curves, soil recovery changes, and mitigation changes can change that maintenance balance. Keep those effects separate in the final interpretation.

### Results from this run

With the source linear equation, `sequential_high_prevention` reached EDI 100 and survived at all three starting-currency levels (500, 1000, and 2000). Its peak normalized pollutant pressure was 0.662, 0.548, and 0.511 respectively. At fixed EDI 100, source high-prevention maintenance survived 200 steps from initial pollution 0 and 10, with peak pressure 0.417 and 0.430. This establishes that EDI 100 is feasible in the source environment under at least one simple policy.

The 3.12 stress candidate with linked-emissions endpoint fraction 0.35 passed the screen, but its effective EDI-100 linked-emissions factor is 1.092, above the source factor of 1.0. It produced higher peak pressure on the same sequential policy (0.828 at starting cash 500, compared with 0.662 in the source case). Because the goal is high EDI with pollution kept low, the source linear formulation is the better reference candidate; the scaled stress case remains a useful boundary test.

The one-step accounting check also resolved the earlier apparent zero-stock anomaly: at source EDI 100, gross air emissions are 11 units, high cleanup removes 10, and 1 unit remains after cleanup. The experiment starts from zero pollution, so it does not produce a zero air stock.

These runs use symmetric regions and one deterministic policy trajectory per cash level, so they establish feasibility but not robustness across regional asymmetry or learned agents. Notebook 17 checks heterogeneous spillover and reward incentives.